In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import GRU, Dense, Dropout, Input, concatenate
from tensorflow.keras.regularizers import l2
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/home/saurabh/core_ai/Variable Input /Merged_Blood_Plasma_Data_sorted.csv')

In [182]:
df['PTETHCAT'].unique()

array(['Not Hisp/Latino', 'Hisp/Latino', 'Unknown'], dtype=object)

In [ ]:
df['PTETHCAT'] = pd.Categorical(df['PTETHCAT'], categories=['Not Hisp/Latino', 'Hisp/Latino', 'Unknown']).codes

In [169]:
df['tr_label'] = df['DX']

In [170]:
df['tr_label'] = pd.Categorical(df['tr_label'], categories=['CN', 'MCI', 'Dementia']).codes

In [4]:
import joblib
try:
    scaler = joblib.load('progression_model/pr-scaler.pkl')
except FileNotFoundError:
        # If scaler is not found, create a new one (for development/testing)
    print("Error while loading scalar")

means = scaler.mean_
stds = scaler.scale_


In [5]:
plasma_features = ["C2N_plasma_ptau217_ratio", 'C2N_plasma_Abeta42', 'C2N_plasma_Abeta42_Abeta40',
                   'C2N_plasma_ptau217', 'Roche_plasma_GFAP', "C2N_plasma_nptau217",
                   'Roche_plasma_NfL', 'AGE', 'MMSE', 'ADAS11', 'ADAS13', 'CDRSB', 'FAQ' , 'tr_label']
demographic_features = ['APOE4', 'PTEDUCAT', 'PTETHCAT', 'PTGENDER']
label_column = 'DX'  # Replace with your label column name

In [172]:
unique_rids = df['RID'].unique()
train_rids, test_rids = train_test_split(unique_rids, test_size=0.1, random_state=42)

In [173]:
train_df = df[df['RID'].isin(train_rids)]
test_df = df[df['RID'].isin(test_rids)]

In [174]:
train_df

,RID,VISCODE2,AGE,MMSE,ADAS11,ADAS13,CDRSB,FAQ,PTGENDER,PTEDUCAT,...,APOE4,C2N_plasma_ptau217_ratio,C2N_plasma_Abeta42,C2N_plasma_Abeta42_Abeta40,C2N_plasma_ptau217,Roche_plasma_GFAP,C2N_plasma_nptau217,Roche_plasma_NfL,DX,tr_label
0,413,m60,76.3,30.0,3.00,4.00,0.0,1.0,0,16,...,0,3.20,46.417,0.10723,1.344,0.1250,42.054,4.78,CN,0
1,413,m84,76.3,30.0,1.00,3.00,0.0,0.0,0,16,...,0,3.96,40.871,0.09746,1.593,0.0928,40.267,4.49,CN,0
2,413,m162,76.3,30.0,5.00,6.00,0.0,0.0,0,16,...,0,2.90,56.211,0.11818,1.595,0.1360,54.922,6.03,CN,0
3,1261,m72,71.1,30.0,8.00,10.00,0.0,0.0,0,16,...,0,4.73,36.742,0.08987,1.975,0.0754,41.751,3.06,CN,0
4,1261,m96,71.1,29.0,8.00,13.00,0.0,0.0,0,16,...,0,4.45,32.295,0.08916,1.946,0.1080,43.708,3.86,CN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,5124,m24,76.7,27.0,11.00,17.00,0.0,0.0,0,20,...,0,3.46,31.933,0.07986,1.814,0.1310,52.375,4.26,CN,0
932,5124,m54,76.7,29.0,8.67,14.67,0.5,0.0,0,20,...,0,4.17,37.008,0.08217,1.879,0.1010,45.093,3.65,CN,0
933,5193,bl,72.5,29.0,4.00,6.00,0.0,0.0,0,16,...,0,1.15,45.083,0.08803,0.650,0.1030,56.557,3.64,CN,0
934,5193,m24,72.5,30.0,6.00,8.00,1.0,0.0,0,16,...,0,1.63,40.439,0.08484,0.650,0.1140,39.872,3.61,CN,0


In [175]:
scaler = StandardScaler()
scaler.fit(train_df[plasma_features])
means = scaler.mean_
stds = scaler.scale_

In [176]:
import pickle
filename = 'pr-scaler.pkl'  # Or any name you prefer
with open(filename, 'wb') as file:  # 'wb' for write binary
    pickle.dump(scaler, file)

In [28]:
def preprocess_inference_data(X_missing, means, stds):
    X_imputed = X_missing.copy()
    
    X_imputed_values = X_imputed.values  
    missing_mask = np.isnan(X_imputed_values) 
    

    for i in range(X_imputed_values.shape[1]):
        X_imputed_values[missing_mask[:, i], i] = means[i] - stds[i]
    X_imputed = pd.DataFrame(X_imputed_values, columns=X_missing.columns)
    
    X_scaled = scaler.transform(X_imputed)
    return X_scaled


In [29]:
def process_data(df, time_steps=2):
    longitudinal_data = []
    demographic_data = []
    labels = []
    
    grouped = df.groupby('RID')
    for rid, group in grouped:
        scaled_plasma = preprocess_inference_data(group[plasma_features], means, stds)
    
        if len(group) < time_steps:
            padded = np.zeros((time_steps, len(plasma_features)))
            padded[:len(group), :] = scaled_plasma
            longitudinal_data.append(padded)
        else:
            longitudinal_data.append(scaled_plasma[:time_steps, :])
    
        demographic_data.append(group[demographic_features].iloc[0].values)
        labels.append(group[label_column].iloc[-1])
    
    return np.array(longitudinal_data), np.array(demographic_data), np.array(labels)

In [30]:
# def process_data(df, scaler, time_steps=2):
#     longitudinal_data = []
#     demographic_data = []
#     labels = []
    
#     grouped = df.groupby('RID')
#     for rid, group in grouped:
#         # Scale plasma features
#         scaled_plasma = scaler.transform(group[plasma_features])
        
#         # Pad/truncate to fixed time_steps
#         if len(group) < time_steps:
#             padded = np.zeros((time_steps, len(plasma_features)))
#             padded[:len(group), :] = scaled_plasma
#             longitudinal_data.append(padded)
#         else:
#             longitudinal_data.append(scaled_plasma[:time_steps, :])
        
#         # Extract static demographic features (first visit)
#         demographic_data.append(group[demographic_features].iloc[0].values)
        
#         # Extract label (last visit)
#         labels.append(group[label_column].iloc[-1])
    
#     return np.array(longitudinal_data), np.array(demographic_data), np.array(labels)

In [73]:
time_steps = 2  # Adjust based on your data
X_train_lon, X_train_dem, y_train = process_data(train_df, time_steps)
X_test_lon, X_test_dem, y_test = process_data(test_df, time_steps)




In [74]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [75]:
num_features = len(plasma_features)  # 13
dem_features = len(demographic_features)  # 4
num_classes = len(le.classes_)  # 3
future_time_s = 1
time_steps = 2

In [76]:
# X_train = X_train_lon
# y_train = np.array(y_train).reshape(-1, 1)
# dem_train = X_train_dem

# X_test = X_test_lon
# y_test = np.array(y_test).reshape(-1, 1)
# dem_test = X_test_dem

In [77]:
X_train = X_train_lon
y_train = y_train
dem_train = X_train_dem

X_test = X_test_lon
y_test = y_test
dem_test = X_test_dem

In [78]:
y_train.shape

(280,)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class LongitudinalModel(nn.Module):
    def __init__(self, num_features, dem_features, num_classes, time_steps):
        super().__init__()
        self.num_features = num_features
        self.dem_features = dem_features
        self.num_classes = num_classes
        self.time_steps = time_steps

        # Longitudinal processing
        self.gru1 = nn.GRU(
            input_size=num_features,
            hidden_size=16,
            batch_first=True
        )
        self.dropout1 = nn.Dropout(0.2)
        self.gru2 = nn.GRU(
            input_size=16,
            hidden_size=8,
            batch_first=True
        )
        self.dropout2 = nn.Dropout(0.2)

        # Demographic processing
        self.classifier = nn.Sequential(
            nn.Linear(8 + dem_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes),
            nn.LogSoftmax(dim=1)  # Added LogSoftmax for stability
        )

    def forward(self, input_lon, input_dem):
        # Mask invalid timesteps
        input_lon = input_lon.masked_fill(input_lon == -1.0, 0)

        # Compute valid timesteps
        mask = (input_lon != 0).any(dim=-1)
        lengths = mask.sum(dim=1)

        # Handle empty sequences
        if (lengths == 0).any():
            raise ValueError("All features masked in some samples")

        # Sort sequences by length
        lengths_sorted, sort_idx = torch.sort(lengths, descending=True)
        input_lon = input_lon[sort_idx]
        input_dem = input_dem[sort_idx]

        # Pack and process through GRUs
        packed = pack_padded_sequence(input_lon, lengths_sorted.cpu(), 
                                      batch_first=True, enforce_sorted=True)

        # First GRU layer
        packed_out, _ = self.gru1(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)
        out = self.dropout1(out)
        out = F.relu(out)

        # Second GRU layer
        packed_out = pack_padded_sequence(out, lengths_sorted.cpu(),
                                           batch_first=True, enforce_sorted=True)
        _, hidden = self.gru2(packed_out)
        x = hidden[-1]  # Get last hidden state
        x = self.dropout2(x)

        # Concatenate with demographic data
        combined = torch.cat([x, input_dem], dim=1)

        # Classification layers
        logits = self.classifier(combined)

        # Restore original order
        _, unsort_idx = torch.sort(sort_idx)
        return logits[unsort_idx]

    def get_regularization_loss(self):
        """L2 regularization for GRU input weights (matching Keras implementation)"""
        reg_loss = 0.0
        l2_lambda = 0.001
        for name, param in self.named_parameters():
            if 'weight' in name:
                reg_loss += l2_lambda * torch.sum(param ** 2)
        return reg_loss


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [107]:
X_train_t = torch.FloatTensor(X_train)
dem_train_t = torch.FloatTensor(dem_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
dem_test_t = torch.FloatTensor(dem_test)
y_test_t = torch.LongTensor(y_test)

In [108]:
class LongitudinalDataset(Dataset):
    def __init__(self, X, dem, y):
        self.X = X
        self.dem = dem
        self.y = y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.dem[idx], self.y[idx]

In [109]:
train_dataset = LongitudinalDataset(X_train_t, dem_train_t, y_train_t)
test_dataset = LongitudinalDataset(X_test_t, dem_test_t, y_test_t)

In [110]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [111]:
print("Original y_train shape:", y_train.shape)
print("Original y_test shape:", y_test.shape)

Original y_train shape: (280,)
Original y_test shape: (32,)


In [112]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LongitudinalModel(
    num_features=X_train.shape[2], 
    dem_features=dem_train.shape[1],
    num_classes=len(np.unique(y_train)),
    time_steps=X_train.shape[1]
).to(device)

In [113]:
class_weights = torch.tensor([0.3, 0.3, 0.4], device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters())

In [114]:
def train_model(model, num_epochs=2):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_dem, batch_y in train_loader:


            batch_X, batch_dem, batch_y = batch_X.to(device), batch_dem.to(device), batch_y.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(batch_X, batch_dem)
            loss = criterion(outputs, batch_y) + model.get_regularization_loss()
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Track metrics
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_X, batch_dem, batch_y in test_loader:
                batch_X, batch_dem, batch_y = batch_X.to(device), batch_dem.to(device), batch_y.to(device)
                
                outputs = model(batch_X, batch_dem)
                loss = criterion(outputs, batch_y) + model.get_regularization_loss()
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()
        
        # Record metrics
        history['train_loss'].append(epoch_loss/len(train_loader))
        history['train_acc'].append(correct/total)
        history['val_loss'].append(val_loss/len(test_loader))
        history['val_acc'].append(val_correct/val_total)
        
        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'Train loss: {history["train_loss"][-1]:.4f}, Acc: {history["train_acc"][-1]:.4f}')
        print(f'Val loss: {history["val_loss"][-1]:.4f}, Acc: {history["val_acc"][-1]:.4f}\n')
    
    return history

In [115]:
history = train_model(model, num_epochs=50)

Epoch 1/50
Train loss: 1.0575, Acc: 0.5143
Val loss: 1.0712, Acc: 0.5938

Epoch 2/50
Train loss: 1.0129, Acc: 0.5107
Val loss: 1.0631, Acc: 0.5938

Epoch 3/50
Train loss: 0.9915, Acc: 0.5214
Val loss: 1.0493, Acc: 0.5938

Epoch 4/50
Train loss: 0.9666, Acc: 0.5286
Val loss: 1.0202, Acc: 0.6562

Epoch 5/50
Train loss: 0.9127, Acc: 0.6714
Val loss: 0.9239, Acc: 0.6875

Epoch 6/50
Train loss: 0.7919, Acc: 0.6964
Val loss: 0.7701, Acc: 0.7188

Epoch 7/50
Train loss: 0.6340, Acc: 0.7536
Val loss: 0.7968, Acc: 0.6250

Epoch 8/50
Train loss: 0.5855, Acc: 0.7607
Val loss: 0.7723, Acc: 0.6875

Epoch 9/50
Train loss: 0.5412, Acc: 0.7893
Val loss: 0.7898, Acc: 0.6562

Epoch 10/50
Train loss: 0.5483, Acc: 0.7714
Val loss: 0.7881, Acc: 0.6875

Epoch 11/50
Train loss: 0.5271, Acc: 0.7964
Val loss: 0.7880, Acc: 0.7188

Epoch 12/50
Train loss: 0.5032, Acc: 0.8000
Val loss: 0.7924, Acc: 0.7812

Epoch 13/50
Train loss: 0.5015, Acc: 0.8036
Val loss: 0.7942, Acc: 0.7500

Epoch 14/50
Train loss: 0.4898, Ac

In [116]:
torch.save(model.state_dict(), "pytorch_progression.pth")


In [98]:
X_train.shape[1]

2

In [8]:
test_model = LongitudinalModel(
    num_features=14, 
    dem_features=4,
    num_classes=3,
    time_steps=2
)

In [9]:
test_model.load_state_dict(torch.load("/home/saurabh/core_ai/Final Application/progression_model/pytorch_progression.pth"))

<All keys matched successfully>

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
test_model.to(device)


LongitudinalModel(
  (gru1): GRU(14, 16, batch_first=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (gru2): GRU(16, 8, batch_first=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=12, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=3, bias=True)
    (5): LogSoftmax(dim=1)
  )
)

(1745, 2, 14)

Testing Model


# Testing on Different Variable outputs. 

In [13]:
def preprocess_inference_data(X_missing, means, stds):
    X_imputed = X_missing.copy()
    
    X_imputed_values = X_imputed.values  
    missing_mask = np.isnan(X_imputed_values) 
    

    for i in range(X_imputed_values.shape[1]):
        X_imputed_values[missing_mask[:, i], i] = means[i] - stds[i]
    X_imputed = pd.DataFrame(X_imputed_values, columns=X_missing.columns)
    
    X_scaled = scaler.transform(X_imputed)
    return X_scaled



In [14]:
def process_data(df, time_steps=2):
    longitudinal_data = []
    demographic_data = []
    labels = []
    
    grouped = df.groupby('RID')
    for rid, group in grouped:
        scaled_plasma = preprocess_inference_data(group[plasma_features], means, stds)
   
        if len(group) < time_steps:
            padded = np.zeros((time_steps, len(plasma_features)))
            padded[:len(group), :] = scaled_plasma
            longitudinal_data.append(padded)
        else:
            longitudinal_data.append(scaled_plasma[:time_steps, :])

        demographic_data.append(group[demographic_features].iloc[0].values)
        labels.append(group[label_column].iloc[-1])
    
    return np.array(longitudinal_data), np.array(demographic_data), np.array(labels)

In [15]:
test_df_ex= pd.read_csv("all_test.csv")

In [16]:
test_df_ex['tr_label'] = test_df_ex['DX']

In [17]:
test_df_ex['tr_label'] = pd.Categorical(test_df_ex['tr_label'], categories=['CN', 'MCI', 'Dementia']).codes

In [18]:
test_df_ex['PTGENDER'] = pd.Categorical(test_df_ex['PTGENDER'], categories=['Female' ,'Male']).codes

In [20]:
time_steps = 2 
X_test_lon_ex , X_test_dem_ex, y_test_ex = process_data(test_df_ex, time_steps)






In [21]:
X_test_lon_ex.shape

(15, 2, 14)

In [22]:
X_test_lon_ex = torch.FloatTensor(X_test_lon_ex)
X_test_dem_ex = torch.FloatTensor(X_test_dem_ex)


In [135]:
device = next(test_model.parameters()).device
X_test_lon_ex = X_test_lon_ex.to(device)
X_test_dem_ex = X_test_dem_ex.to(device)
y_test_ex = y_test_ex.to(device)

In [136]:
test_ex_dataset = torch.utils.data.TensorDataset(X_test_lon_ex, X_test_dem_ex, y_test_ex)
test_ex_loader = torch.utils.data.DataLoader(test_ex_dataset, batch_size=8)

In [138]:
test_model.eval()
all_preds = []
all_probs = []
true_labels = []

In [139]:
with torch.no_grad():
    for batch_X, batch_dem, batch_y in test_ex_loader:
        outputs = test_model(batch_X, batch_dem)
        
        # Get predicted class probabilities
        probs = F.softmax(outputs, dim=1)
        # Get predicted class indices
        _, preds = torch.max(outputs, 1)
        
        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())
        true_labels.append(batch_y.cpu())

In [140]:
all_preds = torch.cat(all_preds).numpy()
all_probs = torch.cat(all_probs).numpy()
true_labels = torch.cat(true_labels).numpy()

In [141]:
all_preds

array([2, 0, 0, 2, 2, 2, 2, 2, 0, 2, 1, 2, 1, 2, 1])

In [142]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"Test Accuracy: {accuracy_score(true_labels, all_preds):.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, all_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, all_preds))

Test Accuracy: 0.7333

Classification Report:
              precision    recall  f1-score   support

         0.0       0.67      0.67      0.67         3
         1.0       1.00      0.60      0.75         5
         2.0       0.67      0.86      0.75         7

    accuracy                           0.73        15
   macro avg       0.78      0.71      0.72        15
weighted avg       0.78      0.73      0.73        15


Confusion Matrix:
[[2 0 1]
 [0 3 2]
 [1 0 6]]


In [156]:
Skipping_plasma = test_df_ex.copy()

In [157]:
Skipping_plasma[['C2N_plasma_ptau217_ratio', 'C2N_plasma_Abeta42' , 'Roche_plasma_GFAP' ]] = np.nan

In [158]:
test_model.eval()
all_preds = []
all_probs = []
true_labels = []

time_steps = 2 
X_test_lon_ex , X_test_dem_ex, y_test_ex = process_data(Skipping_plasma, time_steps)
y_test_ex = le.fit_transform(y_test_ex)

X_test_lon_ex = torch.FloatTensor(X_test_lon_ex)
X_test_dem_ex = torch.FloatTensor(X_test_dem_ex)
y_test_ex = torch.FloatTensor(y_test_ex)

device = next(test_model.parameters()).device
X_test_lon_ex = X_test_lon_ex.to(device)
X_test_dem_ex = X_test_dem_ex.to(device)
y_test_ex = y_test_ex.to(device)

test_ex_dataset = torch.utils.data.TensorDataset(X_test_lon_ex, X_test_dem_ex, y_test_ex)
test_ex_loader = torch.utils.data.DataLoader(test_ex_dataset, batch_size=8)




In [159]:
X_test_lon_ex

tensor([[[-1.0000, -1.0000,  0.9720, -0.7705, -1.0000, -0.6382, -0.6096,
           0.2920, -0.0286, -0.2659, -0.3245,  0.3022, -0.2438,  0.7148],
         [-1.0000, -1.0000,  0.7985, -0.1641, -1.0000, -0.2780, -0.2899,
           0.2920, -0.3986, -0.9377, -0.9068,  0.0144, -0.0412,  0.7148]],

        [[-1.0000, -1.0000, -0.2103, -0.7705, -1.0000, -0.0149, -0.5326,
          -0.1684,  0.3414, -0.2659, -0.3245,  0.0144, -0.4463,  0.7148],
         [-1.0000, -1.0000, -0.4028, -0.7705, -1.0000,  0.0841, -0.4902,
          -0.1684,  0.7114, -0.5463, -0.5190, -0.5613, -0.4463, -0.8612]],

        [[-1.0000, -1.0000,  3.2778, -0.7705, -1.0000,  0.1905, -0.9178,
          -0.4907,  0.7114, -0.4338, -0.6739, -0.5613, -0.4463, -0.8612],
         [-1.0000, -1.0000,  1.5821, -0.7705, -1.0000,  0.6793, -0.5557,
          -0.4907,  0.7114, -0.4892, -0.5959, -0.5613, -0.4463, -0.8612]],

        [[-1.0000, -1.0000,  0.5960, -0.7705, -1.0000, -0.6550, -0.7830,
          -1.4730, -0.7686, -0.0979, -0

In [160]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
with torch.no_grad():
    for batch_X, batch_dem, batch_y in test_ex_loader:
        outputs = test_model(batch_X, batch_dem)
        
        # Get predicted class probabilities
        probs = F.softmax(outputs, dim=1)
        # Get predicted class indices
        _, preds = torch.max(outputs, 1)
        
        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())
        true_labels.append(batch_y.cpu())


all_preds = torch.cat(all_preds).numpy()
all_probs = torch.cat(all_probs).numpy()
true_labels = torch.cat(true_labels).numpy()



print(f"Test Accuracy: {accuracy_score(true_labels, all_preds):.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, all_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, all_preds))

Test Accuracy: 0.7333

Classification Report:
              precision    recall  f1-score   support

         0.0       0.67      0.67      0.67         3
         1.0       1.00      0.60      0.75         5
         2.0       0.67      0.86      0.75         7

    accuracy                           0.73        15
   macro avg       0.78      0.71      0.72        15
weighted avg       0.78      0.73      0.73        15


Confusion Matrix:
[[2 0 1]
 [0 3 2]
 [1 0 6]]


In [163]:
all_probs

array([[9.37499553e-02, 2.34423671e-02, 8.82807672e-01],
       [8.95485222e-01, 1.32432499e-04, 1.04382366e-01],
       [9.54264820e-01, 1.31777906e-05, 4.57220078e-02],
       [3.22989337e-02, 1.12225987e-01, 8.55475128e-01],
       [1.10354484e-03, 2.02884138e-01, 7.96012342e-01],
       [3.83089073e-02, 2.18081567e-02, 9.39882994e-01],
       [6.57782629e-02, 1.64431222e-02, 9.17778671e-01],
       [2.73484848e-02, 5.73639311e-02, 9.15287614e-01],
       [9.29122567e-01, 5.17180852e-05, 7.08257109e-02],
       [7.70136490e-02, 3.71463969e-02, 8.85839880e-01],
       [3.06065413e-05, 7.88118422e-01, 2.11850971e-01],
       [1.35813197e-02, 6.06443211e-02, 9.25774336e-01],
       [9.03554974e-06, 8.44543755e-01, 1.55447185e-01],
       [4.78842203e-03, 1.34272814e-01, 8.60938787e-01],
       [2.28821518e-04, 7.94190884e-01, 2.05580324e-01]], dtype=float32)

In [23]:
def predict_single(lon_data, dem_data):
    # Convert to tensor
    lon_tensor = torch.FloatTensor(lon_data).unsqueeze(0)  # Add batch dim
    dem_tensor = torch.FloatTensor(dem_data).unsqueeze(0)
    
    # Move to device
    lon_tensor = lon_tensor.to(device)
    dem_tensor = dem_tensor.to(device)
    
    # Inference
    model.eval()
    with torch.no_grad():
        output = model(lon_tensor, dem_tensor)
        prob = F.softmax(output, dim=1)
    
    return prob.cpu().numpy().squeeze()

In [167]:
sample_lon = X_test_lon_ex[0].cpu()  # Existing sample
sample_dem = X_test_dem_ex[0].cpu()
probabilities = predict_single(sample_lon, sample_dem)
print(f"Class probabilities: {probabilities}")

Class probabilities: [0.09374996 0.02344238 0.8828077 ]


Skipping C2N_plasma_ptau217_ratio, C2N_plasma_Abeta42 , Roche_plasma_GFAP

In [331]:
Skipping_plasma = test_df_ex.copy()

In [271]:
Skipping_plasma[['C2N_plasma_ptau217_ratio', 'C2N_plasma_Abeta42' , 'Roche_plasma_GFAP' ]] = np.nan

In [272]:
time_steps = 2 
X_test_lon_ex , X_test_dem_ex, y_test_ex = process_data(Skipping_plasma, time_steps)
y_test_ex = le.fit_transform(y_test_ex)
y_test_ex = np.array(y_test_ex).reshape(-1, 1)

In [273]:
print("\nTest Evaluation:")
test_metrics = evaluate_model(model, X_test_lon_ex, X_test_dem_ex, y_test_ex)



Test Evaluation:
Accuracy: 0.4666666666666667
F1 Score (Macro): 0.45396825396825397
Confusion Matrix:
[[3 0 0]
 [0 1 4]
 [4 0 3]]


Skping MMSE , ADAS13, FAQ

In [332]:
Skipping_cognitive = test_df_ex.copy()
Skipping_cognitive[['MMSE', 'ADAS13' , 'FAQ' ]] = np.nan


In [333]:
time_steps = 2 
X_test_lon_ex , X_test_dem_ex, y_test_ex = process_data(Skipping_cognitive, time_steps)
y_test_ex = le.fit_transform(y_test_ex)
y_test_ex = np.array(y_test_ex).reshape(-1, 1)

In [334]:
print("\nTest Evaluation:")
test_metrics = evaluate_model(model, X_test_lon_ex, X_test_dem_ex, y_test_ex)



Test Evaluation:
Accuracy: 0.6
F1 Score (Macro): 0.5079365079365079
Confusion Matrix:
[[3 0 0]
 [0 0 5]
 [1 0 6]]


Skipping C2N_plasma_Abeta42_Abeta40 , C2N_plasma_ptau217 , MMSE

In [335]:
Skipping_cognitive = test_df_ex.copy()
Skipping_cognitive[['ADAS13' , 'FAQ' ]] = np.nan

In [336]:
time_steps = 2 
X_test_lon_ex , X_test_dem_ex, y_test_ex = process_data(Skipping_plasma, time_steps)
y_test_ex = le.fit_transform(y_test_ex)
y_test_ex = np.array(y_test_ex).reshape(-1, 1)

In [337]:
print("\nTest Evaluation:")
test_metrics = evaluate_model(model, X_test_lon_ex, X_test_dem_ex, y_test_ex)



Test Evaluation:
Accuracy: 0.8
F1 Score (Macro): 0.8023809523809525
Confusion Matrix:
[[3 0 0]
 [0 3 2]
 [1 0 6]]
